# Trabajo Final Integrador — Introducción al NLP
## Sistema de recomendación de películas basado en contenido

Dado el corpus `mathigatti/spanish_imdb_synopsis` (~4.970 películas con sinopsis en español, *keywords*, género, año y director) y un conjunto de 14 perfiles de usuario simulados (historial de 5 películas + una *query* en lenguaje natural), el objetivo es generar para cada usuario una lista de **5 películas recomendadas**.

A lo largo del notebook implementamos y comparamos **tres estrategias de representación**:

1. **Embeddings con *Sentence-Transformer*** — concatenamos query + historial en un único texto y lo proyectamos al espacio de embeddings.
2. **Embeddings promediados + LLM** — separamos query e historial, usamos un LLM para clasificar la intención de la query y combinamos ambas señales con un promedio ponderado dinámico.
3. **TF-IDF** — similar a la estrategia anterior pero con vectores tf-idf.

## Instalaciones necesarias

In [2]:
!pip install datasets
!pip install sentence-transformers
!pip install gensim
!pip install bertopic
!pip install unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 5.5 MB/s eta 0:00:00


In [3]:
import pandas as pd
import re
import html
import numpy as np
from collections import Counter
import json as json_lib
import unidecode
from sklearn.feature_extraction.text import TfidfVectorizer

import time

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

## Carga de datasets

Traemos el dataset de sinopsis de peliculas de IMDb desde Hugging Face y el dataset proporcionado con la información de los usuarios y susu respectivas queries

In [4]:
BASE = "https://raw.githubusercontent.com/nazarenomm/Sistema-de-recomendacion-de-peliculas/refs/heads/main/data/"
df_pelis = pd.read_csv(BASE + "peliculas.csv")
usuarios = pd.read_csv(BASE + "usuarios.csv")

Visualizamos las queries

In [5]:
for texto in usuarios['query']:
    print(texto)

Quiero una película donde una mujer enfrenta una amenaza invisible que viene de alguien cercano
Busco algo basado en hechos reales sobre corrupción o poder político
Una comedia donde la relación entre dos personas empieza de forma ridícula o accidental
Algo que haga pensar sobre qué es real y qué es una construcción, con acción pero también ideas
Animación donde el protagonista lucha por su libertad o identidad en un mundo que lo oprime
Un grupo de personas planea un robo o estafa y las cosas se complican de forma inesperada
Una película sobre músicos o artistas que viven al margen, con mucha atmósfera y estilo visual
Acción directa con un héroe que trabaja solo o casi solo contra una organización criminal o corrupta
Algo tranquilo sobre personas que intentan reconectar o entenderse después de una distancia larga
Algo que sea difícil de clasificar, con una lógica narrativa propia, no convencional
No sé bien, algo que valga la pena ver un domingo a la noche, que enganche desde el princi

### Limpieza inicial

Algunos campos vienen con secuencias como `&apos;` en lugar de los caracteres reales. Aplicamos `html.unescape` sobre todas las columnas de tipo texto.

In [6]:
df_pelis.iloc[83].keywords

'escena sexual, relación hermano hermana, reference to guns n&apos; roses, comedia negra, white boy raps'

In [7]:
for col in df_pelis.columns:
    if df_pelis[col].dtype == 'object':  # Solo columnas string
        df_pelis[col] = df_pelis[col].apply(lambda x: html.unescape(str(x)) if pd.notna(x) else x)

In [8]:
df_pelis.iloc[83].keywords

"escena sexual, relación hermano hermana, reference to guns n' roses, comedia negra, white boy raps"

## Preprocesado

Convertimos cada película en un **único texto representativo** que será la entrada de todos los modelos. Esta es una decisión central: qué información del corpus incluimos define qué puede "entender" el sistema por similitud.

### Función de limpieza de texto

Aplicamos un pipeline de limpieza: remover HTML, eliminación de caracteres extraños y espacios múltiples

In [9]:
def limpiar_texto(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'<[^>]+>', ' ', text)        # remover HTML si hubiera
    text = re.sub(r'[^\w\s\.,;:!?áéíóúüñ-]', ' ', text)  # caracteres extraños
    text = re.sub(r'\s+', ' ', text)             # espacios múltiples
    return text.strip()

Unificamos las variables en un texto:
+ Nombre de la película
+ Sinopsis o descripción
+ Director
+ Año de estreno
+ Generos
+ Palabras clave

In [10]:
df_pelis["texto"] = (
    df_pelis["name"].apply(limpiar_texto) + ". "
    + df_pelis["description"].apply(limpiar_texto) + " "
    + df_pelis["director"].fillna('').apply(limpiar_texto) + ". "
    + df_pelis["year"].apply(lambda x: str(int(x)) if pd.notnull(x) else '') + ". "
    + df_pelis["genre"].apply(limpiar_texto) + ". "
    + df_pelis["keywords"].apply(limpiar_texto)
)

In [11]:
df_pelis.texto.iloc[0]

'Herida abierta. Orin Boyd, un duro policía de una comisaría del centro de la ciudad, descubre una red de policías corruptos. Andrzej Bartkowiak. 2001. acción, crimen, suspense. vietnam war veteran, heroína, drogas, narcotraficante, corrupt cop'

#### El spanglish en ``genre`` y ``keywords``:  
El modelo multilingüe que usaremos maneja texto en múltiples idiomas, pero fue entrenado con documentos monolingües por separado, no necesariamente con mezcla de idiomas dentro del mismo string.

No es un problema grave ya que el modelo es bastante robusto esto. Aún así vale la pena mencionar esta limitación del corpus.

## Enfoque 1 - Embeddings con sentence transformer

Primera estrategia de representación. Usamos un modelo `sentence-transformer` multilingüe para proyectar tanto las películas como los usuarios al **mismo espacio vectorial**, y recomendamos por **similitud coseno**.

### Embedding de peliculas

Cargamos el modelo multilingüe

In [12]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

#### Calcular embedding de cada película

Usamos el modelo para calcular los embeddings de cada pelicula en nuestra base de datos usando e texto unificado definido anteriormente.

In [13]:
pelis_embeddings = model.encode(df_pelis['texto'].tolist(), show_progress_bar=True)

Batches:   0%|          | 0/156 [00:00<?, ?it/s]

Como los datasets de películas y embeddings se construyeron en el mismo orden, podemos unirlos directamente por índice.

In [14]:
df_pelis = df_pelis.reset_index(drop=True)

pelis_embeddings_df = pd.DataFrame(pelis_embeddings)
pelis_embeddings_df = pelis_embeddings_df.merge(
    df_pelis[['id', 'name']],
    left_index=True,
    right_index=True
)

In [15]:
pelis_embeddings_df.head()

,0,1,2,3,4,5,6,7,8,9,...,376,377,378,379,380,381,382,383,id,name
0,-0.031084,0.032719,-0.272750,0.105669,0.088466,0.341037,0.109469,0.170736,0.048047,-0.134747,...,-0.053232,0.032655,-0.199530,-0.102709,0.034747,-0.105462,0.243188,0.014966,1,Herida abierta
1,-0.007789,0.076889,-0.077062,0.274105,-0.079885,-0.025508,0.314847,0.059361,0.081432,0.050591,...,0.168026,0.220522,-0.022279,-0.104578,0.325791,-0.082222,-0.040964,-0.080095,2,"Elvira, reina de las tinieblas"
2,-0.017430,0.006371,-0.100607,0.260501,0.053605,0.227207,0.040888,-0.065317,0.184578,-0.041667,...,0.132501,0.318629,-0.022673,0.075589,0.229808,0.034454,0.114800,-0.044366,3,Durmiendo con su enemigo
3,0.137272,0.015918,0.013975,0.060339,-0.133089,0.079739,0.129728,-0.093552,0.063227,0.041110,...,-0.029802,0.355402,0.058575,-0.094542,0.238030,0.101194,0.021751,-0.007258,4,Elizabethtown
4,-0.165777,0.050897,-0.000996,-0.207756,0.106295,-0.271873,-0.016602,0.089710,0.047141,0.133440,...,-0.091462,-0.023140,-0.099317,0.237066,0.205077,-0.302631,0.368318,0.098505,5,Godzilla


Con este dataframe vamos a poder buscar los embedding por nombre de la pelicula o id

### Embeddings de usuarios

Para recomendar necesitamos representar a cada usuario en el **mismo espacio** que las películas. En este primer enfoque construimos el vector de usuario concatenando su *query* con las sinopsis de las 5 películas de su historial, y lo proyectamos con el mismo modelo.

Aplicamos la misma función de limpieza que usamos para las sinopsis. Como la data de usuarios rovista esta limpia no hace falta limpiarlos pero debería ser parte del pipeline operativo habitual.

In [16]:
# data_clean_users = pd.DataFrame(usuarios["query"].apply(limpiar_texto))

Debemos considerar que el modelo tiene limite de 512 tokens. Chequeamos que al pasar el texto ``query + texto unificado del historial`` no superemos este límite.

In [17]:
df_pelis["word_count"] = df_pelis["texto"].apply(lambda x: len(str(x).split()))

print("Cantidad maxima de palabras:")
print(df_pelis["word_count"].max())

print("Mediana de palabras:")
print(df_pelis["word_count"].median())

Cantidad maxima de palabras:
79
Mediana de palabras:
45.0


In [18]:
usuarios["query_word_count"] = usuarios["query"].apply(lambda x: len(str(x).split()))

print("Cantidad maxima de palabras en queries de usuarios:")
print(usuarios["query_word_count"].max())

print("Mediana de palabras en queries de usuarios:")
print(usuarios["query_word_count"].median())

Cantidad maxima de palabras en queries de usuarios:
19
Mediana de palabras en queries de usuarios:
15.5


Igualmente hay que considerar que las queries provistas son cortas, podrían haber más largas en un futuro. Para solucionar esto se podría implementar un límite de palabras para la query de usuario.  
Si suponemos un limite de, por ejemplo, $30$ palabras y que, en promedio, cada palabra se tokeniza en $1.5$-$2$ tokens. Entonces cada texto de usuario tendría:  

``44 palabras × 5 películas + query (max 30 palabras, por ejemplo) ≈ 250 palabras ≈ 375-500 tokens``

Y estaría por debajo del límite.

#### Calcular el embedding de cada usuario

`build_user_text` arma, para cada usuario, un único texto que concatena la **query** con el **texto de las 5 películas del historial**. Ese texto se proyecta con el mismo modelo, de modo que query e historial pesan juntos dentro de un solo vector.


In [19]:
def build_user_text(row, df_pelis):
    # Query del usuario
    partes = [row['query']]

    # Descripciones de las 5 películas del historial
    for col in ['pelicula_1','pelicula_2','pelicula_3','pelicula_4','pelicula_5']:
        nombre = row[col]
        match = df_pelis[df_pelis['name'] == nombre]['texto']
        if len(match):
            partes.append(match.values[0])

    return ' '.join(partes)

In [20]:
user_texts = usuarios.apply(lambda r: build_user_text(r, df_pelis), axis=1).tolist()

Con esos textos unificados y el transformer calculamos el embedding de cada usuario.

In [21]:
user_embeddings = model.encode(user_texts, show_progress_bar=True)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [22]:
user_embeddings_df = pd.DataFrame(user_embeddings)
user_embeddings_df = user_embeddings_df.merge(usuarios[['id']], left_index=True, right_index=True)
user_embeddings_df.head()

,0,1,2,3,4,5,6,7,8,9,...,375,376,377,378,379,380,381,382,383,id
0,-0.238107,-0.128500,-0.207465,0.260500,0.247655,0.290276,0.095238,-0.093639,0.200505,0.013914,...,0.297109,0.074182,0.340460,-0.116549,0.221071,0.159515,-0.142231,0.079386,-0.026740,U01
1,-0.249083,0.191250,-0.289829,0.216814,0.266727,0.115557,0.088643,-0.020128,0.025625,0.020821,...,-0.078047,-0.033263,0.171906,-0.238212,-0.107697,0.169403,-0.052990,-0.165263,-0.162586,U02
2,-0.010083,-0.286078,-0.028388,-0.073591,0.070636,0.379458,0.180789,0.056127,0.217427,-0.043305,...,0.053136,0.069376,0.004023,-0.328375,-0.023175,0.125202,0.085754,0.105665,0.007408,U03
3,-0.135121,-0.022123,-0.174233,-0.013320,0.024583,-0.171468,0.109334,0.055659,0.042445,0.104168,...,-0.050089,-0.095498,0.186531,-0.198212,0.121175,0.162361,0.030514,-0.036455,0.011859,U04
4,-0.022891,0.008990,-0.241990,0.069403,0.142337,0.157824,0.238181,-0.189650,0.251596,0.146036,...,0.137518,0.138678,0.324707,0.010308,0.202955,0.194683,-0.085762,-0.002339,-0.013053,U05


### Recomendaciones y evaluación


Con los vectores de usuarios y películas calculamos la **similitud coseno** entre cada usuario y todas las películas, filtramos las ya vistas (poniendo su score en `-inf`) y nos quedamos con el **top-5**.


Para validar necesitamos algún tipo de etiqueta objetivo. Como no existe un listado de "películas correctas", usamos el **género** como aproximación de relevancia.

Primero miramos qué géneros existen en el corpus y su frecuencia, para entender con qué vocabulario de géneros contamos.

In [23]:
contador_generos = Counter()

for generos_str in df_pelis['genre']:
    generos_str = generos_str.strip('[]')
    generos_list = [g.strip() for g in generos_str.split(',')]
    contador_generos.update(generos_list)

# convertir a DataFrame
df_generos = pd.DataFrame(
    list(contador_generos.items()),
    columns=['Género', 'Frecuencia']
).sort_values('Frecuencia', ascending=False).reset_index(drop=True)

print(df_generos)

             Género  Frecuencia
0             drama        2649
1           comedia        1932
2            acción        1222
3            crimen        1083
4          aventura         944
5           romance         856
6          suspense         740
7            terror         532
8          misterio         479
9          fantasía         412
10         familiar         361
11        animación         351
12  ciencia ficción         347
13        biografía         268
14         historia         151
15           música         148
16          deporte         113
17           bélico         103
18       documental          49
19          musical          37
20            corto          32
21        del oeste          22
22          reality          13
23         concurso           7
24         tertulia           4
25         noticias           2


Miramos los generos del historial de cada usuario

In [24]:
resultados = []

for idx, usuario in usuarios.iterrows():
    usuario_id = usuario['id']
    generos_usuario = Counter()

    for col in ['pelicula_1', 'pelicula_2', 'pelicula_3', 'pelicula_4', 'pelicula_5']:
        nombre_pelicula = usuario[col]
        pelicula = df_pelis[df_pelis['name'] == nombre_pelicula]

        if not pelicula.empty:
            generos_str = pelicula.iloc[0]['genre']
            generos_str = generos_str.strip('[]')
            generos_list = [g.strip() for g in generos_str.split(',')]
            generos_usuario.update(generos_list)

    # obtener top 3-4 géneros por usuario
    top_generos = generos_usuario.most_common(5)
    generos_str = ', '.join([f"{g} ({f})" for g, f in top_generos])

    resultados.append({
        'Usuario': usuario_id,
        'Géneros historial': generos_str
    })

conteo_hist = pd.DataFrame(resultados)
print(conteo_hist.to_string())

   Usuario                                                        Géneros historial
0      U01  suspense (4), drama (3), misterio (3), romance (1), ciencia ficción (1)
1      U02         drama (5), biografía (3), crimen (3), historia (1), misterio (1)
2      U03                                      comedia (5), romance (5), drama (3)
3      U04    ciencia ficción (5), acción (3), drama (2), suspense (2), romance (1)
4      U05         animación (5), aventura (4), drama (2), acción (2), familiar (1)
5      U06             crimen (5), drama (3), comedia (2), acción (1), misterio (1)
6      U07              drama (4), música (3), comedia (3), romance (2), crimen (1)
7      U08          acción (5), aventura (3), crimen (2), suspense (2), comedia (1)
8      U09          drama (5), comedia (3), romance (2), aventura (1), fantasía (1)
9      U10           drama (5), comedia (2), crimen (2), fantasía (1), suspense (1)
10     U11           drama (2), animación (1), aventura (1), acción (1), cri

Y las queries de cada uno

In [25]:
for i in range(len(usuarios)):
    print(f"U{i+1}: " + usuarios["query"].iloc[i])

U1: Quiero una película donde una mujer enfrenta una amenaza invisible que viene de alguien cercano
U2: Busco algo basado en hechos reales sobre corrupción o poder político
U3: Una comedia donde la relación entre dos personas empieza de forma ridícula o accidental
U4: Algo que haga pensar sobre qué es real y qué es una construcción, con acción pero también ideas
U5: Animación donde el protagonista lucha por su libertad o identidad en un mundo que lo oprime
U6: Un grupo de personas planea un robo o estafa y las cosas se complican de forma inesperada
U7: Una película sobre músicos o artistas que viven al margen, con mucha atmósfera y estilo visual
U8: Acción directa con un héroe que trabaja solo o casi solo contra una organización criminal o corrupta
U9: Algo tranquilo sobre personas que intentan reconectar o entenderse después de una distancia larga
U10: Algo que sea difícil de clasificar, con una lógica narrativa propia, no convencional
U11: No sé bien, algo que valga la pena ver un do

A partir de los géneros del historial y de lo que pide cada query, definimos **a ojo** tres géneros esperados para cada usuario. Esto solo es posible para los **9 perfiles "definidos"** (preferencias claras y consistentes); los usuarios ambiguos (U10–U14) no se pueden etiquetar de forma confiable, así que la evaluación cuantitativa se restringe a esos 9.

Con estas etiquetas calculamos, para cada usuario:
- **Recall (géneros):** proporción de géneros esperados que aparecen en el top-5.
- **Precision (películas):** proporción de películas del top-5 que tienen al menos un género esperado.
- **F1:** media armónica de ambas.

In [26]:
etiquetas_a_ojo_def = [
    ["suspense", "terror", "drama"],
    ["crimen", "biografía", "historia"],
    ["comedia", "romance", "drama"],
    ["acción", "ciencia ficción", "suspense"],
    ["animación", "drama", "aventura"],
    ["crimen", "acción", "comedia"],
    ["música", "drama", "comedia"],
    ["acción", "crimen", "aventura"],
    ["drama", "romance", "comedia"],
    # el U10 es ambiguo, la etiqueta debe estar mal
    # los demás usuarios son ambiguos
]

Ahora calculamos la similitud coseno entre los embeddings de usuarios y los de las peliculas, elegimos los 5 más similares y mostramos los resultados para todos los usuarios (perfiles ambiguos incluídos para análisis)

In [27]:
def filtrar_historial(scores, usuarios, df_pelis, hist_cols=['pelicula_1', 'pelicula_2', 'pelicula_3', 'pelicula_4', 'pelicula_5']):
    """
    Setea a -inf el score de las películas que ya están en el historial de cada usuario,
    para que no aparezcan en sus recomendaciones.
    """
    scores_filtrado = scores.copy()

    for i, row in usuarios.iterrows():
        for col in hist_cols:
            nombre = row[col]
            match = df_pelis[df_pelis['name'] == nombre].index
            if len(match):
                scores_filtrado[i, match[0]] = -np.inf

    return scores_filtrado

def generar_recomendaciones_csv(top5_indices, scores_filtrado, usuarios, df_pelis, output_file):
    filas = []
    for i, row in usuarios.iterrows():
        fila = {'id': row['id'], 'nombre': row['nombre']}
        for rank, idx in enumerate(top5_indices[i], start=1):
            pelicula = df_pelis.iloc[idx]
            fila[f'pelicula_{rank}'] = pelicula['name'] + f" ({int(pelicula['year']) if not pd.isna(pelicula['year']) else 'N/A'})"
            fila[f'score_{rank}'] = scores_filtrado[i, idx]
        filas.append(fila)
    resultados_df = pd.DataFrame(filas)
    resultados_df.to_csv(output_file, index=False)
    return resultados_df

In [28]:
scores = cosine_similarity(user_embeddings, pelis_embeddings)

# Filtrar películas del historial seteando su score a -inf
scores_filtrado = filtrar_historial(scores, usuarios, df_pelis)

# Top-5 por usuario
top5_indices = scores_filtrado.argsort(axis=1)[:, -5:][:, ::-1]

resultados_df = generar_recomendaciones_csv(top5_indices, scores_filtrado, usuarios, df_pelis, 'recomendaciones_embeddings.csv')

In [29]:
resultados_df[['id', 'nombre', 'pelicula_1', 'pelicula_2', 'pelicula_3', 'pelicula_4', 'pelicula_5']]

,id,nombre,pelicula_1,pelicula_2,pelicula_3,pelicula_4,pelicula_5
0,U01,Valentina,Tránsito (2006),La mano que mece la cuna (1992),Begotten (1991),Truly Madly Deeply (1992),La novia cadáver (2005)
1,U02,Rodrigo,Historia de un soldado (1985),Monster's Ball (2002),Vida de este chico (1994),Una árida estación blanca (1989),Gorilas en la niebla (1989)
2,U03,Camila,Ahora los padres son ellos (2010),Mira quién habla también (1991),Más allá del odio (2005),Swingers (1997),La familia Addams (1992)
3,U04,Tomás,La red (1995),Pulse (Conexión) (2007),El cortador de césped (1992),Nivel 13 (1999),Proyecto Brainstorm (1983)
4,U05,Lucía,Tránsito (2006),Begotten (1991),Durmiendo con su enemigo (1991),La mano que mece la cuna (1992),Crónica de un engaño (2010)
5,U06,Martín,Como pez en el agua (1998),El negociador (1997),Persecución extrema (2007),Albino Alligator (1997),¡Que te calles! (2004)
6,U07,Sofía,Ray (2005),Hedwig and the Angry Inch (2001),La diva (1981),Miedo y asco en Las Vegas (1999),El corazón del ángel (1987)
7,U08,Diego,Hollywood: Departamento de homicidios (2003),Más fuerte que el odio (1988),The Fast and the Furious (A todo gas) (2001),Max Payne (2008),L.A. Confidential (1997)
8,U09,Elena,Brothers (Hermanos) (2010),Algo en común (2005),El turista accidental (1989),Todos están bien (2010),La vida de Pi (2012)
9,U10,Facundo,Apocalipsis (1994),Seul contre tous (1999),Atando cabos (2002),La última casa a la izquierda (2009),Aún sé lo que hicisteis el último verano (1999)


A simple vista el sistema esta lejos de ser perfecto:
+ Nicolás (ambiguo) pide una película que no sea muy larga y se le recomienda Titanic que dura 3 horas y media.
+ Varios mencionan en sus queries características del film que no suelen estar en las variables dadas en el corpus, pero sí podrían estar presentes en las reseñas de las películas, por ejemplo: "que enganche desde el principio", "lógica narrativa propia" o "con buenas actuaciones".
+ Mariana (ambiguo) pide algo **distinto** a lo de siempre pero el sistema siempre busca similitud con su historial, además su query no aporta información propia.

Para solucionar algunos de estos problemas necesitariamos más información sobre las películas, como su duración o el resumen de reseñas que muestra imdb en su web.
Para solucionar las queries sin peso propio pero que refieren al historial vamos a explorar calcular los embeddings de usuario como un promedio ponderado de su query y su historial. Esto lo veremos en el próximo enfoque.

Corremos la evaluación planteada anteriormente sobre los usuarios con perfiles definidos.

In [30]:
def evaluar_recomendaciones(scores, top5_indices, usuarios, df_pelis, etiquetas_a_ojo_def, output_file, n_usuarios=9):
    """
    Evalúa el top-5 de recomendaciones por usuario contra géneros esperados (a ojo),
    calculando Recall, Precision y F1 por usuario. Imprime el detalle y guarda el CSV.
    """
    resultados_eval = []

    for user_idx, row in usuarios.head(n_usuarios).iterrows():
        print(f"\n{'='*70}")
        print(f"{row['nombre']} ({row['tipo_perfil']})")
        print(f"Query: {row['query']}")
        generos_esperados = set(etiquetas_a_ojo_def[user_idx])
        print(f"Géneros Esperados: {', '.join(generos_esperados)}")
        print(f"{'='*70}")

        generos_recomendados = Counter()
        peliculas_buenas = 0  # con al menos 1 género esperado
        peliculas_malas = []  # sin ningún género esperado

        print("\nTop-5 Recomendaciones:")
        for rank, idx in enumerate(top5_indices[user_idx], 1):
            pelicula = df_pelis.iloc[idx]
            score = scores[user_idx, idx]

            # Extraer géneros
            generos_str = pelicula['genre'].strip('[]')
            generos_list = [g.strip() for g in generos_str.split(',')]
            generos_pelicula = set(generos_list)
            generos_recomendados.update(generos_list)

            # ¿Tiene algún género esperado?
            es_buena = bool(generos_pelicula & generos_esperados)
            if es_buena:
                peliculas_buenas += 1
                marker = "✓"
            else:
                peliculas_malas.append(pelicula['name'])
                marker = "✗"

            print(f"  {rank}. [{marker}] {pelicula['name']} ({int(pelicula['year']) if not pd.isna(pelicula['year']) else 'N/A'}) — {score:.4f}")
            print(f"     Géneros: {', '.join(generos_list)}")

        # Métricas
        generos_capturados = set(generos_recomendados.keys())
        recall = len(generos_capturados & generos_esperados) / len(generos_esperados)
        precision = peliculas_buenas / 5  # top-5
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        print(f"\nMÉTRICAS:")
        print(f"  Recall (géneros):     {recall:.1%}  ({len(generos_capturados & generos_esperados)}/{len(generos_esperados)})")
        print(f"  Precision (películas): {precision:.1%}  ({peliculas_buenas}/5)")
        print(f"  F1-Score:            {f1:.1%}")

        if peliculas_malas:
            print(f"\nPelículas problemáticas (sin géneros esperados):")
            for pelicula in peliculas_malas:
                print(f"    - {pelicula}")

        resultados_eval.append({
            'id': row['id'],
            'Usuario': row['nombre'],
            'Recall': recall,
            'Precision': precision,
            'F1': f1
        })

    df_eval = pd.DataFrame(resultados_eval)
    df_eval.to_csv(output_file, index=False)
    return df_eval

In [31]:
df_eval = evaluar_recomendaciones(scores, top5_indices, usuarios, df_pelis, etiquetas_a_ojo_def, output_file="evaluacion_embeddings.csv")

print(f"\n\n{'='*70}")
print("RESUMEN DE EVALUACIÓN")
print(f"{'='*70}")
print(df_eval.to_string(index=False))
print(f"\nPromedios:")
print(f"  Recall:    {df_eval['Recall'].mean():.1%}")
print(f"  Precision: {df_eval['Precision'].mean():.1%}")
print(f"  F1-Score:  {df_eval['F1'].mean():.1%}")


Valentina (definido)
Query: Quiero una película donde una mujer enfrenta una amenaza invisible que viene de alguien cercano
Géneros Esperados: suspense, terror, drama

Top-5 Recomendaciones:
  1. [✓] Tránsito (2006) — 0.7501
     Géneros: drama, misterio, suspense
  2. [✓] La mano que mece la cuna (1992) — 0.6389
     Géneros: drama, suspense
  3. [✓] Begotten (1991) — 0.6329
     Géneros: fantasía, terror
  4. [✓] Truly Madly Deeply (1992) — 0.6250
     Géneros: comedia, drama, fantasía
  5. [✓] La novia cadáver (2005) — 0.6224
     Géneros: animación, drama, familiar

MÉTRICAS:
  Recall (géneros):     100.0%  (3/3)
  Precision (películas): 100.0%  (5/5)
  F1-Score:            100.0%

Rodrigo (definido)
Query: Busco algo basado en hechos reales sobre corrupción o poder político
Géneros Esperados: crimen, historia, biografía

Top-5 Recomendaciones:
  1. [✓] Historia de un soldado (1985) — 0.5787
     Géneros: crimen, drama, misterio
  2. [✗] Monster's Ball (2002) — 0.5781
     Géneros

+ Valentina: A pesar de que las métricas parecen buenas, ninguna de las peliculas recomendadas cumple al 100% con la query. Esto puede ser porque la película que busca (probablemente "El hombre invisible") no está en la base de datos. Las recomendaciones que están más cerca de cumplir son:
    + La Mano Que Mece La Cuna (1992): una mujer enfrenta una amenaza de alguien cercano, su niñera.
    + Truly Madly Deeply (1992): una mujer protagonista que es visitada por alguien cercano e invisible, el fantasma de su marido.

+ Rodrigo: F1 del 63.2%. Las películas que cumplen en parte con la query pueden ser:
    +  Historia de un soldado (1985): no está basada en hechos reales pero, viendo la descripción, puede tratar sobre corrupción y poder político.
    + Una árida estación blanca (1989): basada en hechos reales y trata sobre corrupción en la policía.

+ Camila: A pesar de las métricas del 100% ninguna película cumple con la consigna de una relación que empiece de forma ridicula o accidental. Esto puede ser porque no hay tal pelicula o porque no se suele mencionar en la sinopsis. El 100% en las métricas se da porque menciona explícitamente que quiere ver una comedia y su historial es consistente con esto.

Con los demás usuarios siguen ocurriendo cosas similares. Con esto queremos dejar en claro que las métricas calculadas no son para nada confiables a la hora de evaluar el sistema ya que hay varios usuarios con métricas perfectas pero a los que se les recomiendan peliculas que no cumplen con sus requisitos.

## Enfoque 2 - Embeddings promediados + LLM

El Enfoque 1 mete query e historial en un mismo texto y no permite **regular el peso** de cada señal. Acá las separamos y las combinamos con un **promedio ponderado**, donde los pesos los decide la **intención de la query**.

Para detectar esa intención usamos un LLM local (`llama3.2` vía Ollama) que clasifica cada query en una de tres categorías, y según la categoría:
- damos más peso a la **query** o al **historial**, y elegimos las películas **más** o **menos** similares (para queries que piden "algo distinto").

**Pipeline del enfoque:**

1. Calcular el embedding de cada **query**.
2. Calcular el embedding del **historial** (promedio de los embeddings de las 5 películas vistas).
3. Clasificar la intención de la query con **Ollama**.
4. Combinar query e historial con un **promedio ponderado** según esa clasificación, y recomendar.

In [32]:
queries_embeddings = model.encode(usuarios['query'].tolist(), show_progress_bar=True)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [33]:
def promedios_historial_embeddings(usuarios_df, pelis_embeddings_df):
    all_embeddings = []

    for _, usuario_row in usuarios_df.iterrows():
        historial_embeddings = []

        for col in ['pelicula_1','pelicula_2','pelicula_3','pelicula_4','pelicula_5']:
            nombre = usuario_row[col]
            peli_emb = pelis_embeddings_df[pelis_embeddings_df['name'] == nombre]
            if not peli_emb.empty:
                historial_embeddings.append(peli_emb.drop(columns=['id', 'name']).values[0])
            else:
                print(f"Película '{nombre}' no encontrada.")

        embedding_promedio = np.mean(historial_embeddings, axis=0)
        all_embeddings.append(embedding_promedio)

    return np.array(all_embeddings)

In [34]:
historial_embeddings = promedios_historial_embeddings(usuarios, pelis_embeddings_df)

Película 'Rec' no encontrada.
Película 'El secreto de sus ojos' no encontrada.
Película 'El exorcista' no encontrada.
Película 'Intocable' no encontrada.
Película 'Una mente brillante' no encontrada.
Película 'Paddington' no encontrada.


Cuando no encontramos una película en nuestra base de datos, simplemente no la contamos en el cálculo del embedding del historial

### Configuración de flash

Usamos Ollama (`llama3.2`) para clasificar cada query en **3 categorías**, cada una con su estrategia de combinación:

1. **`normal`** — query específica que no apela al historial → más peso a la query (`[0.8, 0.2]`) y top-5 más similares.
2. **`historial_positivo`** — el usuario quiere "lo de siempre" o la query es tan vaga que el historial es la mejor guía → más peso al historial (`[0.2, 0.8]`) y top-5 más similares.
3. **`historial_negativo`** — el usuario pide algo **distinto** a lo habitual → más peso al historial pero eligiendo las **menos** similares (`bottom-5`).

El prompt fija la salida en JSON y usa `temperature=0.1` con `seed` fijo para que la clasificación sea reproducible.

In [37]:
from google.colab import ai

MODEL_GEMINI = 'google/gemini-2.5-flash-lite'

CATEGORIA_PARAMS = {
    'normal':                    {'weights': [0.8, 0.2], 'direction': 'top-5'},
    'historial_positivo':        {'weights': [0.2, 0.8], 'direction': 'top-5'},
    'historial_negativo':        {'weights': [0.2, 0.8], 'direction': 'bottom-5'},
    'no_evaluable':               {'weights': [0.2, 0.8], 'direction': 'top-5'},
}

In [38]:
def procesar_query_con_llm(query: str) -> dict:
    """
    Clasifica la query en una de 4 categorías y devuelve los parámetros correspondientes:
      - normal: query específica y evaluable con los datos disponibles
      - historial_positivo: el usuario quiere algo típico o personalizado para él, o la query es vaga
      - historial_negativo: el usuario quiere algo distinto a su costumbre
      - no_evaluable: la query depende de atributos que el sistema no puede medir
    """
    prompt = f"""You are a text classifier for a movie recommendation system.

The system represents each movie using only these fields: plot synopsis, keywords, genre, director, and release year. It does NOT have access to runtime/duration, acting quality, cinematography quality, ratings, or any other subjective or technical attribute not listed above.

Classify the following user query into exactly one of these four categories:

- "normal": the user describes what they want to watch using attributes the system CAN evaluate (genre, mood, theme, plot, time period, director). No reference to their viewing history.
- "historial_positivo": the user wants something similar to what they usually watch, or their request is so vague that their history is the best guide.
- "historial_negativo": the user explicitly wants something DIFFERENT from their usual preferences.
- "no_evaluable": the user's request depends mainly on attributes the system CANNOT evaluate (e.g. acting quality, movie length/runtime, visual effects, soundtrack, pacing) — even if framed as a specific request.

User query: "{query}"

Examples:
"Quiero una película donde un hombre se enfrenta a una organización criminal" → "normal"
"No sé, algo que valga la pena" → "historial_positivo"
"Lo de siempre está bien" → "historial_positivo"
"Quiero algo distinto a lo que vengo viendo" → "historial_negativo"
"Sorpréndeme con algo que no elegiría yo" → "historial_negativo"
"Algo con buenas actuaciones" → "no_evaluable"
"Que sea corta, no tengo mucho tiempo" → "no_evaluable"
"Una comedia con buen ritmo y buenos efectos visuales" → "no_evaluable"

Respond ONLY with valid JSON, no extra text:
{{"categoria": "normal" | "historial_positivo" | "historial_negativo" | "no_evaluable"}}"""

    try:
        raw = ai.generate_text(prompt, model_name=MODEL_GEMINI).strip()
        raw = raw.replace('```json', '').replace('```', '').strip()

        parsed = json_lib.loads(raw)
        categoria = parsed.get('categoria', 'normal').strip()
        if categoria not in CATEGORIA_PARAMS:
            print(f"[WARN] Categoría inválida '{categoria}', usando 'normal'")
            categoria = 'normal'
        params = CATEGORIA_PARAMS[categoria]
        return {
            'categoria': categoria,
            'weights': params['weights'],
            'direction': params['direction']
        }
    except Exception as e:
        print(f"[WARN] Error procesando query '{query[:40]}...': {e}")
        return {'categoria': 'normal', 'weights': [0.8, 0.2], 'direction': 'top-5'}

In [39]:
import time

# Procesar todas las queries
resultados_llm = []
for query in usuarios['query']:
    res = procesar_query_con_llm(query)
    resultados_llm.append(res)
    print(f"Query:      {query}")
    print(f"Categoría:  {res['categoria']} | Weights: {res['weights']} | Dirección: {res['direction']}")
    print()
    time.sleep(4.5)  # evitar rate limit de google.colab.ai

parametros_usuarios = [{'weights': r['weights'], 'direction': r['direction']} for r in resultados_llm]

Query:      Quiero una película donde una mujer enfrenta una amenaza invisible que viene de alguien cercano
Categoría:  normal | Weights: [0.8, 0.2] | Dirección: top-5

Query:      Busco algo basado en hechos reales sobre corrupción o poder político
Categoría:  normal | Weights: [0.8, 0.2] | Dirección: top-5

Query:      Una comedia donde la relación entre dos personas empieza de forma ridícula o accidental
Categoría:  normal | Weights: [0.8, 0.2] | Dirección: top-5

Query:      Algo que haga pensar sobre qué es real y qué es una construcción, con acción pero también ideas
Categoría:  normal | Weights: [0.8, 0.2] | Dirección: top-5

Query:      Animación donde el protagonista lucha por su libertad o identidad en un mundo que lo oprime
Categoría:  normal | Weights: [0.8, 0.2] | Dirección: top-5

Query:      Un grupo de personas planea un robo o estafa y las cosas se complican de forma inesperada
Categoría:  normal | Weights: [0.8, 0.2] | Dirección: top-5

Query:      Una película sobre 

### Análisis de la clasificación con el LLM

La clasificación es **coherente con la redacción de las queries**:

- La mayoría de las queries describen explícitamente qué se quiere ver (género, trama, atmósfera) y caen en **`normal`** (U1, U3, U5, U6, U7, U8, U9, U10, U11, U13).
- **`historial_positivo`** aparece en queries vagas: U4 ("algo que haga pensar…") y U14 ("algo intermedio").
- **`historial_negativo`** lo detecta solo en U12 ("quiero algo distinto a lo de siempre…"), que es exactamente el caso que justifica invertir la búsqueda (`bottom-5`).

**Observaciones / límites:**
- Algunas asignaciones son discutibles (p. ej. clasificar U2 y U4 como `historial_positivo` cuando las query sí son bastante específicas), lo que muestra que el LLM puede fallar en casos de frontera.
- La calidad del enfoque depende de que la clasificación sea correcta: un error de categoría cambia drásticamente los pesos y la dirección de la búsqueda.

### Calculo del embedding de usuario

Para cada usuario combinamos su embedding de query y su embedding de historial con `np.average`, usando los **pesos** que devolvió la clasificación del LLM. El resultado es un vector de usuario que refleja, de forma controlada, cuánto debe pesar lo que pide ahora frente a lo que suele ver.

In [40]:
user_embeddings_pond = []

for i, params in enumerate(parametros_usuarios):
    w = params['weights']
    emb = np.average(
        [queries_embeddings[i], historial_embeddings[i]],
        axis=0,
        weights=w
    )
    user_embeddings_pond.append(emb)

user_embeddings_pond = np.array(user_embeddings_pond)

### Recomendaciones

In [41]:
scores_pond = cosine_similarity(user_embeddings_pond, pelis_embeddings)

# Filtrar películas del historial seteando su score a -inf
scores_pond_filtrado = filtrar_historial(scores_pond, usuarios, df_pelis)

# Top-5 por usuario según dirección
top5_indices_pond = []
for i, params in enumerate(parametros_usuarios):
    if params['direction'] == 'bottom-5':
        # Excluir los -inf del historial también para bottom-5
        scores_validos = scores_pond_filtrado[i].copy()
        scores_validos[scores_validos == -np.inf] = np.inf  # que no aparezcan en el bottom
        indices = scores_validos.argsort()[:5]
    else:
        indices = scores_pond_filtrado[i].argsort()[-5:][::-1]
    top5_indices_pond.append(indices)

top5_indices_pond = np.array(top5_indices_pond)

resultados_df_pond = generar_recomendaciones_csv(top5_indices_pond, scores_pond_filtrado, usuarios, df_pelis, 'recomendaciones_embeddings_pond.csv')

In [42]:
resultados_df_pond[['id', 'nombre', 'pelicula_1', 'pelicula_2', 'pelicula_3', 'pelicula_4', 'pelicula_5']]

,id,nombre,pelicula_1,pelicula_2,pelicula_3,pelicula_4,pelicula_5
0,U01,Valentina,El ente (1983),Tránsito (2006),Femme Fatale (2003),Alone in the Dark (2006),The Ring (La señal) (2003)
1,U02,Rodrigo,Enemigo público (1999),El asesinato de Richard Nixon (2006),La noche cae sobre Manhattan (1997),Dark Blue (2004),Sospechoso (1988)
2,U03,Camila,Escuela de novatos (2004),La pareja basura (1991),Misterioso asesinato en Manhattan (1994),La comedia sexual de una noche de verano (1982),Ensalada de gemelas (1988)
3,U04,Tomás,Proyecto Brainstorm (1983),Pequeños guerreros (1998),El asesinato de Richard Nixon (2006),Paycheck (2004),La gran huida (1984)
4,U05,Lucía,Daredevil (2003),Virtuosity (1995),Número 9 (2010),Code Geass: Lelouch of the Rebellion (2006),Cuentos de Terramar (2006)
5,U06,Martín,Plan oculto (2006),Supercañeras: El internado puede ser una fiest...,Sin motivo aparente (2003),Albino Alligator (1997),Heat (1996)
6,U07,Sofía,Aquarius (1987),Once (2007),Ragtime (1982),El vagón de la muerte (2008),Slacker (1991)
7,U08,Diego,El único (2002),The Punisher (El castigador) (2004),El coche fantástico (N/A),Snake Eyes (Ojos de serpiente) (1998),Una historia de violencia (2005)
8,U09,Elena,El orfanato (2007),A la deriva (2006),Red Rock West (1995),Reeker (2006),La tostadora valiente (1994)
9,U10,Facundo,Inland Empire (2007),El bosque (2004),Misteriosa obsesión (2004),La trama (1998),Proyecto Brainstorm (1983)


### Validación

In [43]:
df_eval_pond = evaluar_recomendaciones(scores_pond_filtrado, top5_indices_pond, usuarios, df_pelis, etiquetas_a_ojo_def, output_file="evaluacion_embeddings_pond.csv")

print(f"\n\n{'='*70}")
print("RESUMEN DE EVALUACIÓN")
print(f"{'='*70}")
print(df_eval_pond.to_string(index=False))
print(f"\nPromedios:")
print(f"  Recall:    {df_eval_pond['Recall'].mean():.1%}")
print(f"  Precision: {df_eval_pond['Precision'].mean():.1%}")
print(f"  F1-Score:  {df_eval_pond['F1'].mean():.1%}")


Valentina (definido)
Query: Quiero una película donde una mujer enfrenta una amenaza invisible que viene de alguien cercano
Géneros Esperados: suspense, terror, drama

Top-5 Recomendaciones:
  1. [✓] El ente (1983) — 0.6280
     Géneros: drama, terror
  2. [✓] Tránsito (2006) — 0.6187
     Géneros: drama, misterio, suspense
  3. [✓] Femme Fatale (2003) — 0.6117
     Géneros: crimen, drama, misterio
  4. [✓] Alone in the Dark (2006) — 0.5942
     Géneros: acción, terror, ciencia ficción
  5. [✓] The Ring (La señal) (2003) — 0.5898
     Géneros: terror, misterio

MÉTRICAS:
  Recall (géneros):     100.0%  (3/3)
  Precision (películas): 100.0%  (5/5)
  F1-Score:            100.0%

Rodrigo (definido)
Query: Busco algo basado en hechos reales sobre corrupción o poder político
Géneros Esperados: crimen, historia, biografía

Top-5 Recomendaciones:
  1. [✗] Enemigo público (1999) — 0.5596
     Géneros: acción, suspense
  2. [✓] El asesinato de Richard Nixon (2006) — 0.5558
     Géneros: biogra

+ Valentina: query común con F1 88.9%, todos los generos esperados fueron capturados. La única película problemática es Scary Movie, que por su título es un error aceptable. Las recomendaciones que más cumplen con lo solicitado:
    + El ente (1983): cumple casi a la perfección con la query. Trata sobre una mujer atacada por un demonio invisible.
    + Femme Fatale (2003): mujer protagonista. Puede que el modelo confunda "su pasado como estafadora regresa para perseguirla" como la amenaza invisible.

+ Rodrigo: Se categorizó mal su query, dandole excesivo peso a su historial. Las recomendaciónes no muestran los géneros biografía ni historia. Aún así no están tan mal, gracias a que su query es bastante consistente con su historial.

+ Camila: query común con F1 de 50%. Todas las películas son comedias pero no se capturan todos los géneros esperados, esto puede ser por una mala clasificación de las películas. Las recomendaciones que más cumplen con lo solicitado:
    + Escuela de novatos (2004): comedia que empieza con dos personajes conociendoce en una situación ridícula.
    + Ensalada de gemelas (1988): comedia donde cuatro personajes se conocen de manera extraña.

Todas las métricas cayeron en relación con el enfoque anterior, pero parecen funcionar de manera similar. Hay que recordar que las métricas están basadas en suposiciones a ojo y que los generos de las películas pueden estar mal asignados (hemos visto varios casos).

## Enfoque 3

### Preprocesado

In [44]:
df_pelis["texto_tfidf"] = (
    df_pelis["name"].apply(limpiar_texto) + ". "
    + df_pelis["description"].apply(limpiar_texto) + " "
    + df_pelis["director"].fillna('').apply(limpiar_texto) + ". "
    # + df_pelis["year"].apply(lambda x: str(int(x)) if pd.notnull(x) else '') + ". "
    + df_pelis["genre"].apply(limpiar_texto) + ". "
    + df_pelis["keywords"].apply(limpiar_texto)
)
df_pelis["texto_tfidf"].iloc[0]

'Herida abierta. Orin Boyd, un duro policía de una comisaría del centro de la ciudad, descubre una red de policías corruptos. Andrzej Bartkowiak. acción, crimen, suspense. vietnam war veteran, heroína, drogas, narcotraficante, corrupt cop'

### Vectorización TF-IDF

Ajustamos el `TfidfVectorizer` sobre el texto de las películas. Decisiones de configuración:

- **`stop_words`**: lista de *stopwords* en español.
- **`ngram_range=(1, 2)`**: incluimos unigramas y bigramas, para capturar expresiones como *"ciencia ficción"* o *"guerra mundial"*.
- **`min_df=2`**: ignoramos términos que aparecen en una sola película (ruido).
- **`max_df=0.5`**: ignoramos términos presentes en más del 50% del corpus (demasiado genéricos).
- **`sublinear_tf=True`**: usa `1 + log(tf)` en vez de `tf`, para que una palabra repetida muchas veces no domine.

In [45]:
stopwords = pd.read_csv('https://raw.githubusercontent.com/gefero/ecyt_lcd_intro_nlp/main/U3/data/stopwords.txt',
                       sep='\t',
                       names=['word'])
stopwords['word'] = stopwords['word'].apply(unidecode.unidecode)

In [46]:
vectorizer = TfidfVectorizer(
    stop_words=stopwords['word'].tolist(),
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.5,
    sublinear_tf=True
)

pelis_tfidf = vectorizer.fit_transform(df_pelis["texto"].tolist())
print("Matriz TF-IDF de películas:", pelis_tfidf.shape)  # (n_pelis, vocabulario)

Matriz TF-IDF de películas: (4967, 20919)


### Ponderación query / historial (estilo Enfoque 2)

Replicamos la idea del **Enfoque 2 (embeddings + LLM)**: calculamos por separado el vector TF-IDF de la **query** y el del **historial** (promedio de las 5 películas vistas), y los combinamos con un **promedio ponderado** según la **intención de la query**.

In [47]:
# Vector TF-IDF de la query y del historial, por separado
query_tfidf = vectorizer.transform(usuarios['query'].tolist()).toarray()

# Historial: promedio de los vectores TF-IDF de las 5 películas vistas
name_to_idx = {name: i for i, name in enumerate(df_pelis['name'])}
hist_cols = ['pelicula_1', 'pelicula_2', 'pelicula_3', 'pelicula_4', 'pelicula_5']
hist_tfidf = []
for _, row in usuarios.iterrows():
    idxs = [name_to_idx[row[c]] for c in hist_cols if row[c] in name_to_idx]
    hist_tfidf.append(np.asarray(pelis_tfidf[idxs].mean(axis=0)).ravel())
hist_tfidf = np.vstack(hist_tfidf)

print("query_tfidf:", query_tfidf.shape, "| hist_tfidf:", hist_tfidf.shape)

query_tfidf: (14, 20919) | hist_tfidf: (14, 20919)


In [48]:
user_tfidf = []

for i, params in enumerate(parametros_usuarios):
    w = params['weights']
    emb = np.average(
        [query_tfidf[i], hist_tfidf[i]],
        axis=0,
        weights=w
    )
    user_tfidf.append(emb)

user_tfidf = np.array(user_tfidf)

### Recomendaciones

In [49]:
scores_tfidf = cosine_similarity(user_tfidf, pelis_tfidf)

# Filtrar películas del historial seteando su score a -inf
scores_tfidf_filtrado = filtrar_historial(scores_tfidf, usuarios, df_pelis)

# Top-5 por usuario según dirección
top5_indices_dinamico_tfidf = []
for i, params in enumerate(parametros_usuarios):
    if params['direction'] == 'bottom-5':
        # Excluir los -inf del historial también para bottom-5
        scores_validos = scores_tfidf_filtrado[i].copy()
        scores_validos[scores_validos == -np.inf] = np.inf  # que no aparezcan en el bottom
        indices = scores_validos.argsort()[:5]
    else:
        indices = scores_tfidf_filtrado[i].argsort()[-5:][::-1]
    top5_indices_dinamico_tfidf.append(indices)

top5_indices_tfidf = np.array(top5_indices_dinamico_tfidf)

resultados_df_tfidf = generar_recomendaciones_csv(top5_indices_tfidf, scores_tfidf_filtrado, usuarios, df_pelis, 'recomendaciones_tfidf.csv')

In [50]:
resultados_df_tfidf[['id', 'nombre', 'pelicula_1', 'pelicula_2', 'pelicula_3', 'pelicula_4', 'pelicula_5']]

,id,nombre,pelicula_1,pelicula_2,pelicula_3,pelicula_4,pelicula_5
0,U01,Valentina,Inseparables (1989),Cabin fever 2 (2011),Los ríos de color púrpura 2: Los ángeles del a...,El hombre sin sombra (2000),Suavemente me mata (2002)
1,U02,Rodrigo,Maleficio (2006),El expreso de Elmira (2008),"Buenas noches, y buena suerte. (2006)",El asesinato de Richard Nixon (2006),Machuca (2004)
2,U03,Camila,Loverboy (1989),Padre Made in USA (2005),Forasteros en Nueva York (1999),Testigo accidental (1990),La posesión (1981)
3,U04,Tomás,Gothika (2004),Halloweentown: ¡Qué familia la mía! (1998),Así se hace (2001),El aceite de la vida (1993),Joint Security Area (JSA) (2000)
4,U05,Lucía,V de Vendetta (2006),Samurai Jack (2017),Haru en el reino de los gatos (2002),Cheque en blanco (1994),Libertad para morir (1991)
5,U06,Martín,Phantoms (1998),Los idiotas (1999),La cena (2010),Pret-a-porter (1995),La rebelión de las máquinas (1986)
6,U07,Sofía,The Shield: Al margen de la ley (2003),Begotten (1991),"Adiós, muchachos (1989)",Los fabulosos Baker Boys (1990),El rey de la comedia (1982)
7,U08,Diego,Las aventuras de Jackie Chan (2000),Justa venganza (2005),Pusher 3: Soy el ángel de la muerte (2005),Air America (1990),Luna negra (1987)
8,U09,Elena,Padre Ted (1995),Algo en común (2005),Click (2006),Bienvenido a Mooseport (2004),South Park (1997)
9,U10,Facundo,Koyaanisqatsi (1984),Innocence (2005),Noviembre dulce (2001),Lars y una chica de verdad (2008),Juno (2008)


### Validación

In [51]:
df_eval_tfidf = evaluar_recomendaciones(scores_tfidf_filtrado, top5_indices_tfidf, usuarios, df_pelis, etiquetas_a_ojo_def, output_file="evaluacion_tfidf.csv")

print(f"\n\n{'='*70}")
print("RESUMEN DE EVALUACIÓN")
print(f"{'='*70}")
print(df_eval_tfidf.to_string(index=False))
print(f"\nPromedios:")
print(f"  Recall:    {df_eval_tfidf['Recall'].mean():.1%}")
print(f"  Precision: {df_eval_tfidf['Precision'].mean():.1%}")
print(f"  F1-Score:  {df_eval_tfidf['F1'].mean():.1%}")


Valentina (definido)
Query: Quiero una película donde una mujer enfrenta una amenaza invisible que viene de alguien cercano
Géneros Esperados: suspense, terror, drama

Top-5 Recomendaciones:
  1. [✓] Inseparables (1989) — 0.1459
     Géneros: drama, terror, suspense
  2. [✓] Cabin fever 2 (2011) — 0.1410
     Géneros: terror
  3. [✗] Los ríos de color púrpura 2: Los ángeles del apocalipsis (2004) — 0.1403
     Géneros: acción, crimen, misterio
  4. [✓] El hombre sin sombra (2000) — 0.1249
     Géneros: acción, terror, ciencia ficción
  5. [✓] Suavemente me mata (2002) — 0.1236
     Géneros: drama, misterio, romance

MÉTRICAS:
  Recall (géneros):     100.0%  (3/3)
  Precision (películas): 80.0%  (4/5)
  F1-Score:            88.9%

Películas problemáticas (sin géneros esperados):
    - Los ríos de color púrpura 2: Los ángeles del apocalipsis

Rodrigo (definido)
Query: Busco algo basado en hechos reales sobre corrupción o poder político
Géneros Esperados: crimen, historia, biografía

Top

## Resultados

**ST y ST+LLM** generaron recomendaciones significativamente mejores que **BERTopic**. Aunque los resultados en ambos casos fueron subóptimos, los embeddings densos mantuvieron coherencia semántica.

### Limitaciones del problema

- **Dataset pequeño con historial escaso**: 14 usuarios con 5 películas cada uno es insuficiente para una validación robusta.
- **Etiquetado subjetivo**: Géneros esperados asignados manualmente, sin consenso.
- **Géneros incorrectos o faltantes**: Los géneros de las películas pueden estar mal asignados o géneros descriptivos pueden estar ausentes.
- **Métricas permisivas**: Una película con un género esperado se considera "correcta", aunque sea una coincidencia parcial.

### Mejoras sugeridas

1. Expandir el dataset de peliculas de imdb con el resumen generado a partir de reseñas presente en la web.
2. Usar un LLM más avanzado que ``llama3.2``, que sea capaz de asignar pesos dinámicos a las queries.
3. Usar dicho LLM para "mejorar" la query y que se parezca más a una sinopsis, mecionando géneros esperados.
4. Cambiar algunas componentes o hiperparametros de BERTopic podría mejorar su performance.


## Conclusión final

Embeddings densos superan a topic modeling en este problema, pero el desempeño general sigue siendo limitado. La mejora requiere datos más ricos y modelos más avanzados.